# Capítulo 2: Probabilidade

Notebook com o **código** deste capítulo, para o Google Colab. Cada trecho vem precedido de uma explicação curta; o texto completo está no site do livro.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem.

In [ ]:
# Setup (rode uma vez).
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

## 2.1 Modelos Probabilísticos

Importa as bibliotecas usadas no capítulo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from formato import num

Simula 60, 600 e 60.000 lançamentos de um dado equilibrado e calcula a frequência relativa de cada face. A semente 42 faz o resultado sair igual ao do livro.

In [ ]:
rng = np.random.default_rng(42)

frequencias = {}
for n in [60, 600, 60_000]:
    lancamentos = rng.integers(1, 7, size=n)   # faces de 1 a 6
    contagem = pd.Series(lancamentos).value_counts(normalize=True)
    frequencias[n] = contagem.sort_index()

tabela = pd.DataFrame(frequencias)
tabela.index.name = "face"
tabela.round(3)

Desenha as três distribuições de frequência lado a lado, com a linha do modelo em 1/6. Com n pequeno as barras oscilam; com n grande ficam coladas na linha.

In [ ]:
fig, eixos = plt.subplots(1, 3, sharey=True, figsize=(9, 3.5))
for eixo, n in zip(eixos, tabela.columns):
    eixo.bar(tabela.index, tabela[n])
    eixo.axhline(1/6, color="black", linestyle="--", linewidth=1)
    eixo.set_title(f"n = {num(n, 0)}")
    eixo.set_xlabel("face")
eixos[0].set_ylabel("frequência relativa")
plt.tight_layout()
plt.show()

Enumera o espaço amostral de três builds, cada um passando (P) ou falhando (F). O `product` faz o produto cartesiano: são $2 \times 2 \times 2 = 8$ sequências.

In [ ]:
omega = ["".join(r) for r in product("PF", repeat=3)]
omega

Representa o modelo do dado como um dicionário (ponto amostral → probabilidade) e os eventos como conjuntos. A função `prob` soma as probabilidades dos pontos de um evento.

In [ ]:
modelo_dado = {face: 1/6 for face in range(1, 7)}

par = {2, 4, 6}
maior_que_3 = {4, 5, 6}

def prob(evento, modelo):
    return sum(modelo[w] for w in evento)

prob(par, modelo_dado), prob(maior_que_3, modelo_dado)

Monta o evento "exatamente dois falham". Um `set` de strings não tem ordem fixa, então o `sorted` deixa a saída sempre igual.

In [ ]:
dois_falham = {w for w in omega if w.count("F") == 2}
sorted(dois_falham)

Carrega os dados dos estados e monta o modelo do sorteio de um estado: 27 pontos amostrais, cada um com probabilidade 1/27.

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")

modelo_uf = {sigla: 1/27 for sigla in estado["Sigla"]}
len(modelo_uf)

Calcula a probabilidade de sortear um estado com mais de 10 milhões de habitantes. Com pontos equiprováveis, ela é a frequência relativa, que o pandas calcula direto com `.mean()`.

In [ ]:
grandes = set(estado.loc[estado["Populacao"] > 10_000_000, "Sigla"])
print(sorted(grandes), prob(grandes, modelo_uf))
print((estado["Populacao"] > 10_000_000).mean())

Calcula a probabilidade de sortear um estado com taxa de homicídios acima da mediana. Dá 13/27, e não 1/2: com $n = 27$ ímpar, a mediana é a taxa de um estado, que não fica acima nem abaixo dela.

In [ ]:
mediana_taxa = estado["Taxa.Homicidios"].median()
acima = set(estado.loc[estado["Taxa.Homicidios"] > mediana_taxa, "Sigla"])
len(acima), prob(acima, modelo_uf)

Troca o modelo: agora sorteamos uma **pessoa**, e a probabilidade de cada estado é a fração da população que mora nela. O evento é o mesmo conjunto de siglas, mas a probabilidade muda.

In [ ]:
modelo_pessoa = dict(zip(estado["Sigla"], estado["Populacao"] / estado["Populacao"].sum()))
prob(grandes, modelo_uf), prob(grandes, modelo_pessoa)